# SRT + Entropy-Augmented Reward — Small Sample Experiment

**Dataset:** DAPO (`ftajwar/deduplicated_dapo_dataset`) — 100 math problems  
**Change from original paper:** $R_{\text{total}} = R_{\text{consistency}} + \alpha \cdot H_{\text{batch}}$  
**Where:** $H_{\text{batch}}$ = normalized Shannon entropy of answer distribution across rollouts

---
### Cell status guide
- 🟢 **Runs locally** — pure Python, no GPU needed  
- 🟡 **Needs API / GPU** — generates model rollouts (skips if results already cached)

## 0 — Imports & Config 🟢

In [31]:
import re, math, json, random, time
from collections import Counter
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datasets

print('Imports OK')

Imports OK


In [4]:
# ── Experiment config ─────────────────────────────────────────────────────────
N_SAMPLES   = 50
N_ROLLOUTS  = 8       # rollouts per problem  (paper uses 16; 8 is faster to test)
TEMPERATURE = 1.0
TOP_P       = 1.0
MAX_TOKENS  = 1024
SEED        = 42

ALPHAS      = [0.0, 0.01, 0.1, 0.5]   # entropy bonus weights to sweep

# ── Lab inference API ─────────────────────────────────────────────────────────
LAB_API_BASE = 'http://100.110.96.82:8000/v1'
LAB_API_KEY  = 'fo7kvpET9v6qMX__8CWFHCuv-LjGhnJriZy0QowkIv8'
MODEL        = 'qwen2.5-32b'     # best math model available on the lab server

RESULTS_FILE = 'rollout_cache.json'   # cache so we don't regenerate rollouts

random.seed(SEED)
np.random.seed(SEED)
print('Config OK')

Config OK


## 1 — Load DAPO Dataset 🟢

In [5]:
ds = datasets.load_dataset('ftajwar/deduplicated_dapo_dataset', split='train')
print(f'DAPO size : {len(ds):,} problems')
print(f'Columns   : {ds.column_names}')
print()

# Sample 100 problems
indices = random.sample(range(len(ds)), N_SAMPLES)
samples = [ds[i] for i in indices]

# Preview one example
ex = samples[0]
print('Example problem:')
print(' prompt :', ex['prompt'][:200], '...')
print(' answer :', ex['answer'])

DAPO size : 17,398 problems
Columns   : ['prompt', 'answer', 'source', 'id']

Example problem:
 prompt : What is the smallest positive integer $n$ such that $20 \equiv n^{15} \pmod{29}$? ...
 answer : 20


In [21]:
INSTRUCTION = "Let's think step by step and output the final answer within \\boxed{}."

def make_prompt(example: dict) -> str:
    return example['prompt'].strip() + '\n\n' + INSTRUCTION

prompts       = [make_prompt(s) for s in samples]
ground_truths = [str(s['answer']).strip() for s in samples]

print(f'Built {len(prompts)} prompts')
print('\nFirst prompt (truncated):')
print(prompts[0][:300])

Built 50 prompts

First prompt (truncated):
What is the smallest positive integer $n$ such that $20 \equiv n^{15} \pmod{29}$?

Let's think step by step and output the final answer within \boxed{}.


## 2 — Reward Functions 🟢

These are pure Python — no model needed.

In [33]:
def extract_answer(text: str) -> Optional[str]:
    """Extract the last \\boxed{...} content from a model response."""
    starts = [m.end() for m in re.finditer(r'\\boxed\{', text)]
    if not starts:
        return None

    start = starts[-1]
    depth = 0
    for i in range(start, len(text)):
        char = text[i]
        if char == '{':
            depth += 1
        elif char == '}':
            if depth == 0:
                return text[start:i].strip()
            depth -= 1
    return None

def normalize(ans: Optional[str]) -> Optional[str]:
    if ans is None:
        return None
    return ans.strip().replace(' ', '').lower()

def answers_match(a: Optional[str], b: Optional[str]) -> bool:
    return normalize(a) == normalize(b) and a is not None

# Quick sanity checks
assert extract_answer('So the answer is \\boxed{42}.')  == '42'
assert extract_answer('\\boxed{\\frac{1}{2}}')           == '\\frac{1}{2}'
assert extract_answer('no box here')                     is None
print('extract_answer: OK')

extract_answer: OK


In [25]:
def majority_vote(responses: list) -> Optional[str]:
    """Majority-voted pseudo-label (the SRT self-consistency step)."""
    answers = [extract_answer(r) for r in responses]
    answers = [a for a in answers if a is not None]
    if not answers:
        return None
    return Counter(answers).most_common(1)[0][0]

def batch_entropy(responses: list) -> float:
    """
    Normalized Shannon entropy of the answer distribution.
      0.0 = all responses give the same answer  (reward hack risk)
      1.0 = all responses give different answers (maximum diversity)
    """
    answers = [normalize(extract_answer(r)) for r in responses]
    answers = [a for a in answers if a is not None]
    if len(answers) <= 1:
        return 0.0
    counts = Counter(answers)
    total  = len(answers)
    probs  = [c / total for c in counts.values()]
    H      = -sum(p * math.log(p) for p in probs if p > 0)
    H_max  = math.log(total)
    return H / H_max if H_max > 0 else 0.0

def srt_reward(responses: list, pseudo_label: Optional[str]) -> list:
    """Original SRT: 1.0 if response matches pseudo-label, else 0.0."""
    if pseudo_label is None:
        return [0.0] * len(responses)
    return [1.0 if answers_match(extract_answer(r), pseudo_label) else 0.0
            for r in responses]

def entropy_reward(responses: list, pseudo_label: Optional[str], alpha: float) -> list:
    """Our change: R_total = R_consistency + alpha * H_batch."""
    base = srt_reward(responses, pseudo_label)
    H    = batch_entropy(responses)
    return [r + alpha * H for r in base]

# Sanity check
fake = ['\\boxed{42}', '\\boxed{42}', '\\boxed{7}', '\\boxed{7}']
print('Entropy (50/50 split):', round(batch_entropy(fake), 4), ' — expected ~1.0')
print('Entropy (all same)   :', batch_entropy(['\\boxed{42}']*4), ' — expected 0.0')
print('SRT reward           :', srt_reward(fake, '42'))
print('Entropy reward α=0.1 :', [round(x,3) for x in entropy_reward(fake, '42', alpha=0.1)])
print('Reward functions: OK')

Entropy (50/50 split): 0.5  — expected ~1.0
Entropy (all same)   : -0.0  — expected 0.0
SRT reward           : [1.0, 1.0, 0.0, 0.0]
Entropy reward α=0.1 : [1.05, 1.05, 0.05, 0.05]
Reward functions: OK


## 3 — Generate Rollouts 🟡

**Needs the lab inference API** (`100.110.96.82:8000`).  
If the API is down or results are already cached, this cell loads from `rollout_cache.json` instead.

In [53]:
from openai import OpenAI

def get_api_client():
    client = OpenAI(base_url=LAB_API_BASE, api_key=LAB_API_KEY, timeout=180)
    try:
        models = [m.id for m in client.models.list().data]
        print('API connected. Available models:', models)
        return client
    except Exception as e:
        print(f'API not available: {e}')
        return None

api_client = get_api_client()

ModuleNotFoundError: No module named 'openai'

In [29]:
def generate_rollouts(client, prompt: str, n: int = N_ROLLOUTS) -> list:
    """Generate n rollouts for one prompt via the lab API."""
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model       = MODEL,
                messages    = [{'role': 'user', 'content': prompt}],
                temperature = TEMPERATURE,
                top_p       = TOP_P,
                max_tokens  = MAX_TOKENS,
                n           = n,
            )
            return [c.message.content for c in resp.choices]
        except Exception as e:
            print(f'  attempt {attempt+1} failed: {e}')
            time.sleep(5)
    return []

print('generate_rollouts defined')

generate_rollouts defined


In [30]:
import os

if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        rollout_data = json.load(f)
    print(f'Loaded {len(rollout_data)} cached results from {RESULTS_FILE}')

elif api_client is not None:
    print(f'Generating rollouts for {N_SAMPLES} problems × {N_ROLLOUTS} rollouts...')
    rollout_data = []
    for i, (prompt, gt) in enumerate(zip(prompts, ground_truths)):
        print(f'  [{i+1:3d}/{N_SAMPLES}] ', end='', flush=True)
        rollouts = generate_rollouts(api_client, prompt)
        if not rollouts:
            print('SKIP')
            continue
        rollout_data.append({'idx': i, 'ground_truth': gt, 'rollouts': rollouts})
        H = batch_entropy(rollouts)
        pseudo = majority_vote(rollouts)
        gt_acc = sum(answers_match(extract_answer(r), gt) for r in rollouts) / len(rollouts)
        print(f'H={H:.2f}  gt_acc={gt_acc:.2f}  pseudo={pseudo}')
    with open(RESULTS_FILE, 'w') as f:
        json.dump(rollout_data, f)
    print(f'\nSaved to {RESULTS_FILE}')

else:
    print('⚠️  API unavailable and no cache found.')
    print('   Start the lab API or run on the GPU server, then re-run this cell.')
    rollout_data = []

NameError: name 'api_client' is not defined

## 4 — Analysis 🟢

All analysis runs on the cached rollouts — no model needed.

In [ ]:
if not rollout_data:
    print('No rollout data — skipping analysis. Run Step 3 first.')
else:
    records = []
    for row in rollout_data:
        rollouts = row['rollouts']
        gt       = row['ground_truth']
        pseudo   = majority_vote(rollouts)
        H        = batch_entropy(rollouts)
        gt_acc   = sum(answers_match(extract_answer(r), gt) for r in rollouts) / len(rollouts)
        pseudo_correct = answers_match(pseudo, gt) if pseudo else False

        rec = {
            'ground_truth'   : gt,
            'pseudo_label'   : pseudo,
            'pseudo_correct' : pseudo_correct,
            'entropy'        : H,
            'gt_accuracy'    : gt_acc,
            'rollouts'       : rollouts,
        }
        # Compute reward per rollout for each alpha
        for alpha in ALPHAS:
            rewards = entropy_reward(rollouts, pseudo, alpha)
            rec[f'mean_reward_alpha{alpha}'] = np.mean(rewards)

        records.append(rec)

    df = pd.DataFrame([{k: v for k, v in r.items() if k != 'rollouts'} for r in records])
    print(f'Problems analysed : {len(df)}')
    print(df[['entropy', 'gt_accuracy', 'pseudo_correct']].describe().round(3))

In [ ]:
if rollout_data:
    print('='*55)
    print('SUMMARY')
    print('='*55)
    print(f"Ground-truth accuracy (per rollout)  : {df['gt_accuracy'].mean():.3f}")
    print(f"Pseudo-label accuracy (majority vote) : {df['pseudo_correct'].mean():.3f}")
    print(f"Mean batch entropy                    : {df['entropy'].mean():.3f}")
    zero_e = (df['entropy'] < 0.05).sum()
    print(f"Problems with entropy≈0 (collapse)    : {zero_e}/{len(df)}")
    print()
    print(f"{'Alpha':>6}  {'Mean R_total':>12}  {'SRT part':>9}  {'H bonus':>8}")
    print(f"  {'-'*6}  {'-'*12}  {'-'*9}  {'-'*8}")
    for alpha in ALPHAS:
        col = f'mean_reward_alpha{alpha}'
        mean_r  = df[col].mean()
        srt_r   = df['mean_reward_alpha0.0'].mean()
        h_bonus = mean_r - srt_r
        tag = '  ← baseline (original SRT)' if alpha == 0.0 else ''
        print(f"  {alpha:>6.2f}  {mean_r:>12.4f}  {srt_r:>9.4f}  {h_bonus:>8.4f}{tag}")

## 5 — Plots 🟢

In [ ]:
if not rollout_data:
    print('No data to plot.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(
        f'SRT Entropy Experiment  |  DAPO {len(df)} problems × {N_ROLLOUTS} rollouts  |  {MODEL}',
        fontsize=11
    )

    # Plot 1: Entropy distribution
    axes[0].hist(df['entropy'], bins=20, color='steelblue', edgecolor='white')
    axes[0].axvline(df['entropy'].mean(), color='red', linestyle='--',
                    label=f"mean={df['entropy'].mean():.2f}")
    axes[0].set_xlabel('Batch entropy H')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Answer diversity per problem')
    axes[0].legend()

    # Plot 2: Pseudo-reward vs true accuracy (reward hack signal)
    srt_per_problem = df['mean_reward_alpha0.0']
    axes[1].scatter(srt_per_problem, df['gt_accuracy'], alpha=0.5, s=15, color='darkorange')
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    axes[1].set_xlabel('SRT pseudo-reward (self-consistency)')
    axes[1].set_ylabel('True accuracy (ground truth)')
    axes[1].set_title('Reward hack check\n(points below diagonal = inflated reward)')
    axes[1].set_xlim(-0.05, 1.05)
    axes[1].set_ylim(-0.05, 1.05)

    # Plot 3: Alpha sweep
    mean_totals = [df[f'mean_reward_alpha{a}'].mean() for a in ALPHAS]
    colors = ['#4c72b0', '#55a868', '#c44e52', '#8172b2']
    axes[2].bar([str(a) for a in ALPHAS], mean_totals, color=colors)
    axes[2].axhline(mean_totals[0], color='gray', linestyle='--',
                    linewidth=0.8, label='baseline (α=0)')
    axes[2].set_xlabel('Alpha (entropy bonus weight)')
    axes[2].set_ylabel('Mean R_total per rollout')
    axes[2].set_title('Effect of entropy bonus on total reward')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig('srt_entropy_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved to srt_entropy_results.png')